# Subject-driven generation on FLUX.1-dev

The released subject-driven LoRAs (`omini/subject_512`) were trained on **FLUX.1-dev**; running them on dev just requires *real image guidance* (below). They also work on **FLUX.1-schnell** without it (see `subject.ipynb`).

There are **two** distinct CFG knobs in `generate()`, and they play different roles:

1. **Image guidance** — `image_guidance_scale`: real classifier-free guidance of the *condition* against an empty (black) condition. On dev you **must** set this `> 1.0`. It runs an extra forward pass and extrapolates: `noise_pred = unc_pred + image_guidance_scale * (noise_pred - unc_pred)`. Without it (the schnell default `image_guidance_scale=1.0`), dev tends to ignore the condition. **This is the only tunable CFG knob here.**
2. **Embedding / distilled guidance** — `guidance_scale`: feeds the model's distilled `guidance_embeds` (dev has `guidance_embeds=True`). This **must be exactly `3.5`** — the value used during training. Fixing it at `3.5` keeps train/inference consistent; it is **not** a free hyperparameter and changing it is not supported.

Also note FLUX.1-dev is **not** timestep-distilled, so it needs more steps than schnell (use ~20–28, not 8).

In short: **schnell** = `image_guidance_scale=1.0`, few steps; **dev** = `image_guidance_scale>1.0` (tunable) + `guidance_scale=3.5` (fixed, required), more steps.

In [ ]:
import os

os.chdir("..")

In [ ]:
import torch
from diffusers.pipelines import FluxPipeline
from PIL import Image

from omini.pipeline.flux_omini import Condition, generate, seed_everything

In [ ]:
pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev", torch_dtype=torch.bfloat16
)
pipe = pipe.to("cuda")
pipe.load_lora_weights(
    "Yuanshi/OminiControl",
    weight_name=f"omini/subject_512.safetensors",
    adapter_name="subject",
)

## Settings for FLUX.1-dev

The values below were validated on this repo (subject `omini/subject_512` LoRA on FLUX.1-dev):

| Parameter | Value (dev) | Tunable? | schnell (for reference) |
| --- | --- | --- | --- |
| `guidance_scale` | **3.5 (required, fixed)** | No — must match training | n/a (schnell ignores it) |
| `image_guidance_scale` | **1.5** (try 1.3–2.0) | Yes — the one CFG knob to tune | 1.0 (default, no image guidance) |
| `num_inference_steps` | **24** (try 20–28) | Yes | 8 |
| resolution | 512 x 512 | — | 512 x 512 |

- `guidance_scale` must be **exactly 3.5**: that is the embedding/distilled-guidance value used during training, so fixing it keeps train/inference consistent. It is not a free hyperparameter — do not change it.
- `image_guidance_scale` is the only CFG knob to tune. Higher values (1.8–2.0) preserve the subject more strongly but can slightly over-saturate; 1.5 is a good default.
- The image-guidance path runs **two** transformer forward passes per step, so dev is ~2x slower per step than the schnell recipe.

In [ ]:
image = Image.open("assets/rc_car.jpg").convert("RGB").resize((512, 512))

# For this model, the position_delta is (0, 32).
# For more details of position_delta, please refer to:
# https://github.com/Yuanshi9815/OminiControl/issues/89#issuecomment-2827080344
condition = Condition(image, "subject", position_delta=(0, 32))

prompt = "A film style shot. On the moon, this item drives across the moon surface. The background is that Earth looms large in the foreground."

seed_everything(0)

result_img = generate(
    pipe,
    prompt=prompt,
    conditions=[condition],
    num_inference_steps=24,
    height=512,
    width=512,
    # FLUX.1-dev specific settings:
    guidance_scale=3.5,  # REQUIRED: must be 3.5 to match training (do not change)
    image_guidance_scale=1.5,  # the tunable CFG knob; must be >1.0 on dev
).images[0]

concat_image = Image.new("RGB", (1024, 512))
concat_image.paste(condition.condition, (0, 0))
concat_image.paste(result_img, (512, 0))
concat_image

In [ ]:
image = Image.open("assets/penguin.jpg").convert("RGB").resize((512, 512))

condition = Condition(image, "subject", position_delta=(0, 32))

prompt = "On Christmas evening, on a crowded sidewalk, this item sits on the road, covered in snow and wearing a Christmas hat."

seed_everything(0)

result_img = generate(
    pipe,
    prompt=prompt,
    conditions=[condition],
    num_inference_steps=24,
    height=512,
    width=512,
    guidance_scale=3.5,
    image_guidance_scale=1.5,
).images[0]

concat_image = Image.new("RGB", (1024, 512))
concat_image.paste(condition.condition, (0, 0))
concat_image.paste(result_img, (512, 0))
concat_image

## Notes

- If the condition seems ignored on dev, the most common cause is leaving `image_guidance_scale=1.0` (the schnell default). Raise it to ~1.5.
- If results look noisy or under-cooked, increase `num_inference_steps` toward 28 — dev is not timestep-distilled.
- The same guidelines as schnell apply: input images are center-cropped/resized to 512x512, and prompts should refer to the subject with phrases like `this item`, `the object`, or `it`.